# cellmap-flow on Colab (production dashboard, no tunnels)

Runs the real `cellmap_flow_app` (the dashboard Flask app) and
`cellmap_flow_server` (the inference Flask app) on this Colab session.
Uses Colab's built-in **port proxy** (`google.colab.kernel.proxyPort`)
to expose both — no cloudflared, no static frontend.

**Limit**: the proxyPort URLs only work for **your own Colab session**.
You cannot share them with colleagues. For a publicly-shareable demo
you'd need to switch to cloudflared (see the older notebook revisions
on this branch).

**IMPORTANT**: after running cell 1 (Install), use **Runtime → Restart
session** before running the rest. The install upgrades numpy, and Colab's
default kernel already has the old numpy imported.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Run cell 1 (Install) and wait for it to finish.
3. **Runtime → Restart session** (NOT "Disconnect runtime" — `Restart
   session` specifically). Required: the install upgrades numpy, and you
   need to restart the Python interpreter to load the new version.
4. Run all remaining cells (skip cell 1 since it's already installed).
3. Click the printed Dashboard URL.
4. The dashboard's NG iframe should load, raw + inference layers
   pre-configured. Adjust Input / Postprocess in the right sidebar and
   click Submit All — changes flow through the real production
   `/api/process` route in the same kernel.


## 1. Install

In [ ]:
# Colab ships numpy 1.26; modern scipy/skimage need numpy>=2. Upgrade
# first so cellmap-flow's deps don't import-fail later.
%pip install -q --force-reinstall "numpy>=2,<3"

%pip install -q "cellmap-flow[bioimageio] @ git+https://github.com/janelia-cellmap/cellmap-flow.git@browser-inference" huggingface_hub s3fs

# Pin bioimageio.spec to a version that still parses v0.4-style RDFs.
%pip install -q --force-reinstall "bioimageio.core==0.9.6" "bioimageio.spec==0.5.7.4"


## 2. Configure model + dataset

`MODEL_TYPE = "huggingface"` for a cellmap HF model, or `"bioimage"`
for a BMZ model.

T4 sizing notes:
- 178³ HF models (`fly_organelles_run07_*`): fit cleanly.
- 288³ HF models (`jrc_mus-livers_*`): borderline, OOM-prone.
- 2D BMZ models (`hiding-blowfish`): trivial, fits anywhere.


In [ ]:
MODEL_TYPE = "huggingface"   # or "bioimage"

# --- Mode A: huggingface ---
HF_REPO = "cellmap/fly_organelles_run07_432000"
HF_NAME = HF_REPO.split("/")[-1]
HF_DATASET = (
    "https://janelia-cosem-datasets.s3.amazonaws.com/"
    "jrc_mus-liver/jrc_mus-liver.zarr/recon-1/em/fibsem-uint8"
)

# --- Mode B: bioimage (BMZ) ---
BMZ_MODEL = "hiding-blowfish"
BMZ_VOXEL_SIZE = "8,8,8"
BMZ_DATASET = (
    "https://janelia-cosem-datasets.s3.amazonaws.com/"
    "jrc_hela-2/jrc_hela-2.zarr/recon-1/em/fibsem-uint8/s1"
)

INFERENCE_PORT = 8765
DASHBOARD_PORT = 8501

if MODEL_TYPE == "huggingface":
    MODEL_NAME = HF_NAME
    DATASET = HF_DATASET
elif MODEL_TYPE == "bioimage":
    MODEL_NAME = BMZ_MODEL
    DATASET = BMZ_DATASET
else:
    raise ValueError(f"unknown MODEL_TYPE={MODEL_TYPE!r}")

print(f"MODEL_TYPE = {MODEL_TYPE}")
print(f"MODEL_NAME = {MODEL_NAME}")
print(f"DATASET    = {DATASET}")


## 3. Start the cellmap-flow inference server (subprocess)

In [ ]:
import os, subprocess, time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

if MODEL_TYPE == "huggingface":
    cmd = [
        "cellmap_flow_server", "huggingface",
        "--repo", HF_REPO, "--name", HF_NAME,
        "-d", HF_DATASET, "--port", str(INFERENCE_PORT),
    ]
elif MODEL_TYPE == "bioimage":
    cmd = [
        "cellmap_flow_server", "bioimage",
        "--model-name", BMZ_MODEL, "--voxel-size", BMZ_VOXEL_SIZE,
        "--name", BMZ_MODEL,
        "-d", BMZ_DATASET, "--port", str(INFERENCE_PORT),
    ]
print("starting:", " ".join(cmd))
server = subprocess.Popen(
    cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    env={**os.environ},
)
print(f"server pid={server.pid}, waiting for it to listen on :{INFERENCE_PORT} ...")
for _ in range(300):
    line = server.stdout.readline()
    if not line: time.sleep(0.25); continue
    print(line, end="")
    if "Running on" in line or f":{INFERENCE_PORT}" in line:
        print("\n[server] ready.")
        break


## 4. Colab proxyPort the inference server

Returns a URL like `https://localhost-XXXX.colab.googleapis.com/` that
only works for your Colab session but is reachable from the NG iframe.

In [ ]:
from google.colab.output import eval_js
INFERENCE_URL = eval_js(f"google.colab.kernel.proxyPort({INFERENCE_PORT})").rstrip("/")
print(f"INFERENCE_URL = {INFERENCE_URL}")


## 5. Pre-populate dashboard state + start the dashboard

Sets up `g.viewer`, `g.dataset_path`, `g.jobs` so the dashboard renders
NG with raw + inference layers without needing manual Set-Data /
Submit-Models clicks. Then runs the dashboard Flask in a thread so the
kernel stays interactive.

In [ ]:
import neuroglancer, threading
from cellmap_flow.globals import g
from cellmap_flow.dashboard.app import app
from cellmap_flow.dashboard import state

# neuroglancer's Python Viewer auto-detects Colab and uses proxyPort
# for its own URL. Just need to bind it to a routable interface first.
neuroglancer.set_server_bind_address("0.0.0.0")

g.dataset_path = DATASET
viewer = neuroglancer.Viewer()
g.viewer = viewer
state.NEUROGLANCER_URL = str(viewer)
print(f"NEUROGLANCER iframe URL: {state.NEUROGLANCER_URL}")

# Pre-populate g.jobs so the dashboard's /api/process knows about the
# (already running) inference server. host points at the Colab-proxied
# URL so the NG iframe in the user's browser can reach it.
class FakeJob:
    def __init__(self, model_name, host):
        self.model_name = model_name
        self.host = host
g.jobs.append(FakeJob(model_name=MODEL_NAME, host=INFERENCE_URL))

# Seed the viewer with raw + inference layers so the iframe shows
# something on first load (before any /api/process call).
with viewer.txn() as s:
    s.dimensions = neuroglancer.CoordinateSpace(
        names=["z", "y", "x"], units="nm", scales=[8, 8, 8],
    )
    import re
    raw_url = re.sub(r"/s\d+/?$", "", DATASET)
    s.layers["data"] = get_raw_layer(raw_url)
    s.layers[MODEL_NAME] = neuroglancer.ImageLayer(
        source=f"zarr://{INFERENCE_URL}/{MODEL_NAME}/",
    )

# Run dashboard Flask in a background thread.
dash_thread = threading.Thread(
    target=lambda: app.run(host="0.0.0.0", port=DASHBOARD_PORT,
                            threaded=True, use_reloader=False, debug=False),
    daemon=True,
)
dash_thread.start()
import time as _t; _t.sleep(2)
print(f"dashboard listening on :{DASHBOARD_PORT}")


## 6. Open the dashboard

In [ ]:
DASHBOARD_URL = eval_js(f"google.colab.kernel.proxyPort({DASHBOARD_PORT})").rstrip("/")
print()
print("=" * 70)
print(f"DASHBOARD URL (open in this browser; only works for your Colab session):")
print(f"  {DASHBOARD_URL}")
print()
print(f"NEUROGLANCER iframe URL (embedded in the dashboard):")
print(f"  {state.NEUROGLANCER_URL}")
print()
print(f"INFERENCE server URL (used by NG to fetch chunks):")
print(f"  {INFERENCE_URL}")
print("=" * 70)


## 7. Keep-alive

Stop with ▢ to tear down. Drains server logs as they come in.

In [ ]:
import time, select

def drain(proc, label):
    while True:
        r, _, _ = select.select([proc.stdout], [], [], 0)
        if not r: return
        line = proc.stdout.readline()
        if not line: return
        print(f"[{label}] {line}", end="")

try:
    while True:
        drain(server, "server")
        if server.poll() is not None:
            drain(server, "server")
            print(f"\n[server] exited rc={server.returncode}.")
            break
        time.sleep(2)
finally:
    try: server.terminate()
    except Exception: pass
